In [ ]:
!pip install -q transformers==4.45.0 peft==0.13.0 accelerate==0.34.2 sentencepiece gradio PyPDF2

import transformers, peft
print("Transformers:", transformers.__version__)
print("PEFT        :", peft.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.8 MB/s eta 0:00:00
Transformers: 4.45.0
PEFT        : 0.13.0


In [ ]:
# Load model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

REPO_ID = "BSVGK/phi35-mini-lora-text2kg-merged"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    REPO_ID,
    trust_remote_code = True,
    token             = HF_TOKEN,
    padding_side      = "left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("Tokenizer loaded")

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype         = torch.float16,
    device_map          = "auto",
    trust_remote_code   = True,
    attn_implementation = "eager",
    token               = HF_TOKEN,
)
model.eval()
model.config.use_cache = True
print("Model loaded. Memory:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

Loading tokenizer...


Tokenizer loaded
Loading model...


A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Model loaded. Memory: 7.64 GB


In [ ]:
# Inference and file handling functions

import torch
import csv
import tempfile
import os
import PyPDF2
import pandas as pd
from datetime import datetime

history_store = []

def run_inference(contract_text):
    if not contract_text.strip():
        return ""
    try:
        prompt = f"""<|user|>
Extract entities and relationships from the construction contract description and represent them as triples.

Contract Description:
{contract_text.strip()}<|end|>
<|assistant|>
"""
        inputs = tokenizer(
            prompt,
            return_tensors = "pt",
            truncation     = True,
            max_length     = 800
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens     = 200,
                do_sample          = False,
                repetition_penalty = 1.1,
                pad_token_id       = tokenizer.pad_token_id,
                eos_token_id       = tokenizer.eos_token_id,
            )

        input_length = inputs.input_ids.shape[1]
        return tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens = True
        ).strip()
    except Exception as e:
        return f"Error: {str(e)}"

def extract_text_from_pdf(pdf_path):
    try:
        text = ""
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() + "\n"
        return text.strip()
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

def load_file_to_textbox(file_obj):
    if file_obj is None:
        return ""
    try:
        file_path = file_obj if isinstance(file_obj, str) else file_obj.name
        ext       = os.path.splitext(file_path)[1].lower()
        if ext == ".pdf":
            return extract_text_from_pdf(file_path)
        elif ext == ".csv":
            df       = pd.read_csv(file_path)
            text_col = None
            for col in ['input', 'contract', 'text', 'description', 'contract_description']:
                if col in df.columns:
                    text_col = col
                    break
            if text_col is None:
                text_col = df.columns[0]
            return "\n\n".join(df[text_col].astype(str).tolist())
        else:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
    except Exception as e:
        return f"Error: {str(e)}"

def save_to_csv(rows):
    tmp = tempfile.NamedTemporaryFile(
        mode='w', suffix='.csv', delete=False, newline='', encoding='utf-8'
    )
    writer = csv.writer(tmp)
    writer.writerow(['record_id', 'input', 'prediction', 'timestamp'])
    for i, (text, result, ts) in enumerate(rows):
        writer.writerow([i + 1, text, result, ts])
    tmp.close()
    return tmp.name

def save_to_txt(rows):
    tmp = tempfile.NamedTemporaryFile(
        mode='w', suffix='.txt', delete=False, encoding='utf-8'
    )
    for i, (text, result, ts) in enumerate(rows):
        tmp.write(f"Record {i + 1} — {ts}\n")
        tmp.write(f"Input:\n{text}\n\n")
        tmp.write(f"Prediction:\n{result}\n")
        tmp.write("=" * 60 + "\n\n")
    tmp.close()
    return tmp.name

def generate_output(contract_text, file_obj, download_format):
    if file_obj is not None and not contract_text.strip():
        contract_text = load_file_to_textbox(file_obj)
    if not contract_text.strip():
        return "", "No input provided.", None, get_history_table()

    records = [r.strip() for r in contract_text.split("\n\n") if r.strip()]
    if len(records) == 1:
        records = [contract_text.strip()]

    rows    = []
    outputs = []
    for i, record in enumerate(records):
        result = run_inference(record)
        ts     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        rows.append((record, result, ts))
        outputs.append(result)
        history_store.append({
            'id'        : len(history_store) + 1,
            'input'     : record[:80] + "..." if len(record) > 80 else record,
            'prediction': result[:100] + "..." if len(result) > 100 else result,
            'timestamp' : ts,
        })

    display  = outputs[0] if len(outputs) == 1 else "\n\n".join([f"Record {i+1}:\n{o}" for i, o in enumerate(outputs)])
    tmp_path = save_to_csv(rows) if download_format == "CSV" else save_to_txt(rows)
    status   = f"Done — {len(rows)} record(s) at {datetime.now().strftime('%H:%M:%S')}"
    return display, status, gr.update(value=tmp_path, visible=True), get_history_table()

def get_history_table():
    if not history_store:
        return []
    return [[h['id'], h['timestamp'], h['input'], h['prediction']] for h in history_store]

def clear_history():
    history_store.clear()
    return []

def clear_all():
    return "", None, "", gr.update(value=None, visible=False), "Ready"

print("All functions ready")

All functions ready


In [ ]:
import gradio as gr

css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

html, body {
    background: #eef0f5 !important;
    font-family: 'Inter', sans-serif !important;
}

.gradio-container {
    max-width: 100% !important;
    width: 100% !important;
    margin: 0 !important;
    padding: 0 0 32px !important;
    background: #eef0f5 !important;
    font-family: 'Inter', sans-serif !important;
}

/* Header */
.app-header {
    background: #1a2035;
    padding: 16px 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    min-height: 64px;
    margin-bottom: 0;
}

.app-header-title {
    font-size: 20px;
    font-weight: 700;
    color: #ffffff;
    letter-spacing: -0.01em;
}

.app-header-sub {
    font-size: 12px;
    color: rgba(255,255,255,0.4);
    font-family: 'JetBrains Mono', monospace;
    margin-top: 3px;
}

/* Tabs */
.tab-nav {
    background: transparent !important;
    border-bottom: 1.5px solid #d1d9e6 !important;
    padding: 0 20px !important;
    margin-bottom: 16px !important;
}

.tab-nav button {
    background: transparent !important;
    border: none !important;
    border-bottom: 2.5px solid transparent !important;
    margin-bottom: -1.5px !important;
    color: #718096 !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 13px !important;
    font-weight: 500 !important;
    padding: 10px 18px !important;
    cursor: pointer !important;
    border-radius: 0 !important;
    transition: color 0.15s !important;
}

.tab-nav button.selected {
    color: #e07b35 !important;
    border-bottom-color: #e07b35 !important;
}

.tab-nav button:hover:not(.selected) { color: #4a5568 !important; }

/* Content padding */
.content-pad { padding: 0 20px; }

/* Two column row */
.main-row {
    display: flex !important;
    gap: 16px !important;
    align-items: flex-start !important;
    padding: 0 20px !important;
}

/* Panel cards */
.panel-card {
    background: #1a2035;
    border-radius: 12px;
    padding: 18px;
    display: flex;
    flex-direction: column;
    gap: 10px;
    flex: 1;
}

.panel-title {
    font-size: 14px;
    font-weight: 600;
    color: rgba(255,255,255,0.75);
    padding-bottom: 10px;
    border-bottom: 1px solid rgba(255,255,255,0.07);
    font-family: 'Inter', sans-serif;
    flex-shrink: 0;
}

/* Textareas */
textarea {
    background: #0d1117 !important;
    border: 1px solid rgba(255,255,255,0.07) !important;
    border-radius: 8px !important;
    color: rgba(255,255,255,0.82) !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 13px !important;
    line-height: 1.75 !important;
    padding: 14px !important;
    resize: vertical !important;
    transition: border-color 0.2s !important;
    width: 100% !important;
}

textarea:focus {
    border-color: rgba(49,130,206,0.45) !important;
    outline: none !important;
}

textarea::placeholder {
    color: rgba(255,255,255,0.16) !important;
    font-style: italic !important;
}

.out-area textarea {
    color: #63b3ed !important;
    background: #0d1117 !important;
    border-color: rgba(49,130,206,0.12) !important;
    font-size: 12.5px !important;
    line-height: 1.9 !important;
}

.status-box textarea {
    background: #0d1117 !important;
    border: 1px solid rgba(255,255,255,0.06) !important;
    border-radius: 6px !important;
    color: #68d391 !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px !important;
    padding: 8px 12px !important;
    resize: none !important;
    min-height: 36px !important;
}

/* Labels */
label > span {
    color: rgba(255,255,255,0.35) !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 10px !important;
    font-weight: 600 !important;
    letter-spacing: 0.06em !important;
    text-transform: uppercase !important;
}

.hide-label > label > span { display: none !important; }

/* File upload */
.gr-file, .gr-upload {
    background: #0d1117 !important;
    border: 1px dashed rgba(255,255,255,0.1) !important;
    border-radius: 8px !important;
}

/* Radio */
.gr-radio label {
    color: rgba(255,255,255,0.6) !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 12px !important;
}

input[type="radio"] { accent-color: #3182ce !important; }

/* Buttons */
button.primary {
    background: #3182ce !important;
    color: #fff !important;
    border: none !important;
    border-radius: 7px !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 13px !important;
    font-weight: 600 !important;
    padding: 11px 0 !important;
    width: 100% !important;
    cursor: pointer !important;
    transition: all 0.18s !important;
}

button.primary:hover {
    background: #2b6cb0 !important;
    box-shadow: 0 4px 16px rgba(49,130,206,0.35) !important;
}

button.secondary {
    background: rgba(255,255,255,0.05) !important;
    color: rgba(255,255,255,0.4) !important;
    border: 1px solid rgba(255,255,255,0.09) !important;
    border-radius: 7px !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 12px !important;
    font-weight: 500 !important;
    padding: 10px 0 !important;
    width: 100% !important;
    cursor: pointer !important;
    transition: all 0.15s !important;
}

button.secondary:hover {
    background: rgba(255,255,255,0.09) !important;
    color: rgba(255,255,255,0.7) !important;
}

/* Download */
.gr-file-preview {
    background: rgba(255,255,255,0.03) !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 6px !important;
}

/* History */
.gr-dataframe th {
    background: #1a2035 !important;
    color: rgba(255,255,255,0.45) !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 11px !important;
    font-weight: 600 !important;
    letter-spacing: 0.08em !important;
    text-transform: uppercase !important;
    padding: 10px 14px !important;
}

.gr-dataframe td {
    color: rgba(255,255,255,0.6) !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px !important;
    padding: 8px 14px !important;
    border-bottom: 1px solid rgba(255,255,255,0.04) !important;
    background: #0d1117 !important;
}

.history-hint {
    font-family: 'Inter', sans-serif;
    font-size: 13px;
    color: rgba(255,255,255,0.35);
    margin-bottom: 12px;
    padding: 0 20px;
}

footer { display: none !important; }
"""

with gr.Blocks(css=css, title="Text to Knowledge Graph") as demo:

    # Compact header
    gr.HTML("""
    <div class="app-header">
        <div>
            <div class="app-header-title">Text to Knowledge Graph</div>
            <div class="app-header-sub">
                Phi-3.5 Mini Instruct &nbsp;·&nbsp; LoRA fine-tuned &nbsp;·&nbsp; UK construction contracts
            </div>
        </div>
    </div>
    """)

    with gr.Tabs():

        with gr.Tab("KG Extraction"):
            with gr.Row(equal_height=False, elem_classes=["main-row"]):

                # Left - input panel
                with gr.Column(scale=1):
                    with gr.Column(elem_classes=["panel-card"]):
                        gr.HTML('<div class="panel-title">Input Contract Description</div>')

                        text_input = gr.Textbox(
                            label        = "",
                            placeholder  = "Enter your contract description here...",
                            lines        = 8,
                            max_lines    = 15,
                            elem_classes = ["hide-label"],
                        )

                        file_input = gr.File(
                            label      = "Upload File",
                            file_types = [".csv", ".pdf", ".txt"],
                            type       = "filepath",
                        )

                        load_btn = gr.Button(
                            "Load Uploaded File Into Textbox",
                            variant = "secondary",
                        )

                        download_format = gr.Radio(
                            choices = ["CSV", "TXT"],
                            value   = "CSV",
                            label   = "Download Format",
                        )

                        with gr.Row():
                            generate_btn = gr.Button("Generate Output", variant="primary")
                            clear_btn    = gr.Button("Clear", variant="secondary")

                # Right - output panel
                with gr.Column(scale=1):
                    with gr.Column(elem_classes=["panel-card"]):
                        gr.HTML('<div class="panel-title">Generated KG Triples</div>')

                        output_box = gr.Textbox(
                            label        = "",
                            lines        = 12,
                            max_lines    = 20,
                            interactive  = False,
                            placeholder  = "Your extracted Knowledge Graph triples will appear here...\n\n( subject ,  predicate ,  object )\n( subject ,  predicate ,  object )\n...",
                            elem_classes = ["hide-label", "out-area"],
                        )

                        download_file = gr.File(
                            label   = "Download Result File",
                            visible = False,
                        )

                        status_box = gr.Textbox(
                            label        = "Status",
                            value        = "Ready",
                            interactive  = False,
                            lines        = 1,
                            max_lines    = 1,
                            elem_classes = ["status-box"],
                        )

        with gr.Tab("History"):
            gr.HTML('<div class="history-hint">All extractions from this session are stored here.</div>')

            history_table = gr.Dataframe(
                headers  = ["ID", "Timestamp", "Input Preview", "Prediction Preview"],
                datatype = ["number", "str", "str", "str"],
                value    = [],
                label    = "",
                wrap     = True,
            )

            with gr.Row(elem_classes=["content-pad"]):
                clear_history_btn = gr.Button("Clear History", variant="secondary")

    # Event bindings
    load_btn.click(fn=load_file_to_textbox,   inputs=[file_input],  outputs=[text_input])
    generate_btn.click(fn=generate_output,    inputs=[text_input, file_input, download_format], outputs=[output_box, status_box, download_file, history_table])
    clear_btn.click(fn=clear_all,             outputs=[text_input, file_input, output_box, download_file, status_box])
    clear_history_btn.click(fn=clear_history, outputs=[history_table])

demo.launch(share=True, quiet=True)

/tmp/ipykernel_993/2054541550.py:260: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="Text to Knowledge Graph") as demo:


* Running on public URL: https://2e3bccfc700c8c46a4.gradio.live
